# Error angle and distance to bead data from smooth data

In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Save error angle and distance data csv

In [8]:

# Input / output paths
INPUT_DIR  = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\xyz_Smooth"
OUTPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Error_Angles"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------
# Plane projections
# ----------------------------------------------------------
def project_xy(v):
    return v[:, [0, 1]]

def project_yz(v):
    return v[:, [1, 2]]

# ----------------------------------------------------------
# Signed Error Calculation (Using arctan2 logic)
# ----------------------------------------------------------
def compute_signed_error_2d(forward_2d, target_vec_2d):
    """
    Calculates the signed angle between a heading vector and a target vector.
    Uses arctan2 to determine if the target is to the left or right.
    """
    # Angle of the fly's heading relative to global axis
    heading_angle = np.arctan2(forward_2d[:, 1], forward_2d[:, 0])
    
    # Angle of the target relative to global axis
    target_angle = np.arctan2(target_vec_2d[:, 1], target_vec_2d[:, 0])
    
    # Difference (Target - Heading)
    error = target_angle - heading_angle
    
    # Wrap to [-pi, pi] to ensure the fly takes the shortest angular path
    error = (error + np.pi) % (2 * np.pi) - np.pi
    
    return np.degrees(error)

def compute_metrics(bead, head, center):
    # Calculate forward heading based on velocity vector
    forward = np.gradient(head, axis=0)
    v_bead_from_head = bead - head
    v_heading_from_center = head - center
    v_bead_from_head_metrics = bead - head

    # --- 1. Head-Centric Error Angles (Signed) ---
    # Absolute 3D angle (magnitude only)
    dot_h = np.einsum("ij,ij->i", forward, v_bead_from_head)
    nf = np.linalg.norm(forward, axis=1)
    nb_h = np.linalg.norm(v_bead_from_head, axis=1)
    cos_h = np.clip(dot_h / (nf * nb_h), -1, 1)
    err_h_3d = np.degrees(np.arccos(cos_h))

    # Signed XY (Horizontal)
    err_h_xy = compute_signed_error_2d(project_xy(forward), project_xy(v_bead_from_head))
    
    # Signed YZ (Vertical)
    err_h_yz = compute_signed_error_2d(project_yz(forward), project_yz(v_bead_from_head))

    # --- 2. Center-Centric Error Angles (Signed) ---
    # Absolute 3D angle
    dot_c = np.einsum("ij,ij->i", v_heading_from_center, v_bead_from_head)
    nh_c = np.linalg.norm(v_heading_from_center, axis=1)
    cos_c = np.clip(dot_c / (nh_c * nb_h), -1, 1)
    err_c_3d = np.degrees(np.arccos(cos_c))

    # Signed XY
    err_c_xy = compute_signed_error_2d(project_xy(v_heading_from_center), project_xy(v_bead_from_head))
    
    # Signed YZ
    err_c_yz = compute_signed_error_2d(project_yz(v_heading_from_center), project_yz(v_bead_from_head))

    return err_h_3d, err_h_xy, err_h_yz, err_c_3d, err_c_xy, err_c_yz

# ----------------------------------------------------------
# Main Processing Loop
# ----------------------------------------------------------
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))

for path in csv_files:
    fname = os.path.basename(path)
    print(f"Processing: {fname}")

    df = pd.read_csv(path)
    
    # Extract coordinates
    bead   = df[["pt1_X","pt1_Y","pt1_Z"]].values
    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values

    # Calculate Distances
    dist_center = np.linalg.norm(bead - center, axis=1)
    dist_head   = np.linalg.norm(bead - head, axis=1)

    # Calculate Error Angles
    h_3d, h_xy, h_yz, c_3d, c_xy, c_yz = compute_metrics(bead, head, center)

    # Compile Results
    out_df = pd.DataFrame({
        "frame": df["frame"].values,
        "dist_center_m": dist_center,
        "dist_head_m": dist_head,
        "error_angle_head_deg": h_3d,
        "error_angle_head_xy_deg": h_xy, # Signed
        "error_angle_head_yz_deg": h_yz, # Signed
        "error_angle_center_deg": c_3d,
        "error_angle_center_xy_deg": c_xy, # Signed
        "error_angle_center_yz_deg": c_yz   # Signed
    })

    # Save Output
    out_name = fname.replace("_SMOOTH.csv", "_CHASE_METRICS.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)
    print(f"Saved: {out_name}")

print("\nAll Chase Metrics calculated and saved successfully.")

Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_CHASE_METRICS.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_CHASE_METRICS.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_CHASE_METRICS.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_CHASE_METRICS.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_CHASE_METRICS.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_400rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial5_400rpmxyzpts_CHASE_METRICS.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial7_400rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_

#save and display plots from error angle and distance to bead

In [16]:

# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Error_Angles"
BASE_TRIAL_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data"

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_CHASE_METRICS.csv")))

if not csv_files:
    raise FileNotFoundError("No CHASE_METRICS CSV files found.")

print(f"Found {len(csv_files)} metric files.\n")

# ==========================================================
# STYLE SETTINGS
# ==========================================================
LABEL_SIZE = 18
TICK_SIZE = 14
TITLE_SIZE = 20
LEGEND_SIZE = 14

# ==========================================================
# LOOP THROUGH FILES
# ==========================================================
for path in csv_files:

    base = os.path.splitext(os.path.basename(path))[0]
    print("Plotting:", base)

    m = re.search(r"(Trial\d+_\d+rpm)", base)
    if not m:
        print("Skipping file (Trial+RPM not found)")
        continue

    trial_name = m.group(1)

    trial_dir = os.path.join(BASE_TRIAL_DIR, trial_name)
    save_dir = os.path.join(trial_dir, "CHASE_PLOTS")
    os.makedirs(save_dir, exist_ok=True)

    df = pd.read_csv(path)
    t = df["frame"].values

    # ======================================================
    # ================= CENTER PLOTS =======================
    # ======================================================

    # --- CENTER: Distance vs Resultant ---
    fig, ax1 = plt.subplots(figsize=(14, 6))

    ax1.plot(t, df["dist_center_m"], color="darkred", linewidth=2)
    ax1.set_xlabel("Frame", fontsize=LABEL_SIZE)
    ax1.set_ylabel("Distance (m)", color="darkred", fontsize=LABEL_SIZE)
    ax1.tick_params(axis="both", labelsize=TICK_SIZE)
    ax1.tick_params(axis="y", labelcolor="darkred")
    ax1.grid(True)

    ax2 = ax1.twinx()
    ax2.plot(t, df["error_angle_center_deg"], color="black", linewidth=2)
    ax2.set_ylabel("Resultant Error (°)", color="black", fontsize=LABEL_SIZE)
    ax2.tick_params(axis="both", labelsize=TICK_SIZE)
    ax2.tick_params(axis="y", labelcolor="black")

    plt.title("CENTER: Distance vs Resultant Error", fontsize=TITLE_SIZE)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{base}_CENTER_distance_resultant.png"), dpi=300)
    plt.close()

    # --- CENTER: Components ONLY ---
    plt.figure(figsize=(14, 6))

    plt.plot(t, df["error_angle_center_xy_deg"],
             color="purple", linewidth=2, label="Horizontal (XY)")
    plt.plot(t, df["error_angle_center_yz_deg"],
             color="teal", linewidth=2, label="Vertical (YZ)")

    plt.xlabel("Frame", fontsize=LABEL_SIZE)
    plt.ylabel("Error Angle (°)", fontsize=LABEL_SIZE)
    plt.xticks(fontsize=TICK_SIZE)
    plt.yticks(fontsize=TICK_SIZE)
    plt.title("CENTER: Error Components", fontsize=TITLE_SIZE)
    plt.legend(fontsize=LEGEND_SIZE)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{base}_CENTER_components.png"), dpi=300)
    plt.close()

    # ======================================================
    # ================== HEAD PLOTS ========================
    # ======================================================

    # --- HEAD: Distance vs Resultant ---
    fig, ax1 = plt.subplots(figsize=(14, 6))

    ax1.plot(t, df["dist_head_m"], color="goldenrod", linewidth=2)
    ax1.set_xlabel("Frame", fontsize=LABEL_SIZE)
    ax1.set_ylabel("Distance (m)", color="goldenrod", fontsize=LABEL_SIZE)
    ax1.tick_params(axis="both", labelsize=TICK_SIZE)
    ax1.tick_params(axis="y", labelcolor="goldenrod")
    ax1.grid(True)

    ax2 = ax1.twinx()
    ax2.plot(t, df["error_angle_head_deg"], color="black", linewidth=2)
    ax2.set_ylabel("Resultant Error (°)", color="black", fontsize=LABEL_SIZE)
    ax2.tick_params(axis="both", labelsize=TICK_SIZE)
    ax2.tick_params(axis="y", labelcolor="black")

    plt.title("HEAD: Distance vs Resultant Error", fontsize=TITLE_SIZE)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{base}_HEAD_distance_resultant.png"), dpi=300)
    plt.close()

    # --- HEAD: Components ONLY (BOTH COMPONENTS) ---
    plt.figure(figsize=(14, 6))

    plt.plot(t, df["error_angle_head_xy_deg"],
             color="purple", linewidth=2, label="Horizontal (XY)")
    plt.plot(t, df["error_angle_head_yz_deg"],
             color="teal", linewidth=2, label="Vertical (YZ)")

    plt.xlabel("Frame", fontsize=LABEL_SIZE)
    plt.ylabel("Error Angle (°)", fontsize=LABEL_SIZE)
    plt.xticks(fontsize=TICK_SIZE)
    plt.yticks(fontsize=TICK_SIZE)
    plt.title("HEAD: Error Components", fontsize=TITLE_SIZE)
    plt.legend(fontsize=LEGEND_SIZE)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{base}_HEAD_components.png"), dpi=300)
    plt.close()

    print("Saved plots to:", save_dir)

print("\nAll CENTER and HEAD plots saved successfully.")

Found 7 metric files.

Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_CHASE_METRICS
Saved plots to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_180rpm\CHASE_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_CHASE_METRICS
Saved plots to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_200rpm\CHASE_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_CHASE_METRICS
Saved plots to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial3_180rpm\CHASE_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_CHASE_METRICS
Saved plots to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial4_400rpm\CHASE_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_CHASE_METRICS
Saved plots to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial5_180rpm\CHASE_PLOTS
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial5_400rpmxyzpts_CHASE_METRICS
Saved plots to: D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial5_